# 5. Resultados da programação não linear

Leitura da bateria: o que cada método consegue na PNL cinemática, e como isso
se compara ao paper (\(t_f\approx 1{,}180\) s, obstáculo com \(N=3\)).


In [ ]:
from pathlib import Path
import sys
here = Path.cwd().resolve()
for cand in [here, here.parent, here / "T2", here.parent / "T2"]:
    if (cand / "src" / "parametros.py").exists():
        if str(cand) not in sys.path:
            sys.path.insert(0, str(cand))
        break
from src.ambiente import preparar
T2 = preparar()
print("T2 =", T2)


In [ ]:
import pickle
from src.paths import saida_dir

with open(saida_dir() / "bateria.pkl", "rb") as f:
    payload = pickle.load(f)
resumo = payload["resumo"]
melhor = payload["melhor"]

print(f"{'método':20s} {'ok':>6s} {'tf min':>8s} {'tf méd':>8s} {'tf máx':>8s} {'CPU méd':>8s}")
for m, s in resumo.items():
    print(f"{m:20s} {s['n_sucesso']}/{s['n']:<3d} {s['tf_min']:8.4f} {s['tf_media']:8.4f} "
          f"{s['tf_max']:8.4f} {s['cpu_media']:8.3f}")
print(f"\nmelhor: {melhor['metodo']} / {melhor['chute']}  tf={melhor['tf']:.4f} s")
print("paper, cinemática com obstáculo N=3: tf ≈ 1.180 s")


## Leitura

- **Pontos interiores** (`trust-constr` + polimento SQP) param em um ponto KKT
  nas quatro partidas viáveis. A Jacobiana analítica das restrições entra no
  solver; o polimento fecha o critério de ótimo local que a barreira sozinha
  não atinge neste problema bang-bang.
- **Pattern search** (COBYLA) também converge nas quatro partidas viáveis: a
  parada é o raio mínimo da região de confiança, não o teto de avaliações. Sem
  derivadas o \(t_f\) local fica maior (\(\approx 2{,}4\)–\(2{,}8\) s).
- **Newton-SQP** (`SLSQP`) fecha KKT nas mesmas quatro partidas, com
  \(t_f\approx 1{,}194\) s. A reta no espaço das juntas atravessa o disco e os
  três métodos falham ali (não há direção viável de descida a partir desse chute).
- **Paper.** \(t_f\approx 1{,}18\) s com colocação LGR e malha bem mais fina.
  O melhor desta transcrição, \(t_f\approx 1{,}192\) s, está na mesma ordem.
  A passagem é pela esquerda do disco no C-space, como no caso \(N=3\).
